<a href="https://colab.research.google.com/github/ShinAsakawa/ShinAsakawa.github.io/blob/master/2026notebooks/2026_0511psycholing_metrics_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 準備作業，必要なライブラリのインストール
try:
    import jamorasep
except ImportError:
    !pip install jamorasep
    import jamorasep

try:
    import Levenshtein
except ImportError:
    !pip install Levenshtein
    import Levenshtein

# 開発したファイルを入手
!wget https://raw.githubusercontent.com/ShinAsakawa/CDP_ja/main/psycholing_metrics_ja_v4.py -O psycholing_metrics_ja.py

import psycholing_metrics_ja

In [ ]:
%%time
# 初期から設定する場合，このセルを実行すると所要時間 3 分以上。直下のセルを実行するか，このセルを実行するか，どちらかを選択
import os
wlsp_path = 'bunruidb.txt'
if not os.path.exists(wlsp_path):
    print(f"{wlsp_path} をダウンロード中...")
    import urllib.request
    url = ("https://raw.githubusercontent.com/masayu-a/WLSP/master/bunruidb.txt")
    urllib.request.urlretrieve(url, wlsp_path)
    print("ダウンロード完了")
engine_wlsp = psycholing_metrics_ja.PsychoLingEngine(wlsp_path=wlsp_path)
engine = engine_wlsp

In [ ]:
%%time
# ローカル PC からアップロードする場合，このセルを実行すると所要時間 4 分以上
from google.colab import files
import pickle

# アップロード待機状態になります
uploaded = files.upload()

# ファイル名を指定して読み込み
file_name = list(uploaded.keys())[0]
with open(file_name, 'rb') as f:
    data = pickle.load(f)

print(f"{file_name} を読み込みました。")

with open('engine_psylex.pkl', 'rb') as f:
    engine_psylex = pickle.load(f)
engine = engine_psylex

In [3]:
# ── 心理言語学変数の計算 ──
test_pairs = [
    ('研究', 'ケンキュウ'),  ('国語', 'コクゴ'),
    ('会社', 'カイシャ'),    ('データ', 'データ'),
    ('コース', 'コース'),    ('独立行政法人', 'ドクリツギョウセイホウジン'),
    ('発表', 'ハッピョウ'),  ('熱心', 'ネッシン'),
    ('生活', 'セイカツ'),    ('先生', 'センセイ'),
]

df = engine.compute_batch(test_pairs, verbose=False)
df

,word,reading_normalized,n_chars,n_morae,ons,pns,pns_segment,old20,pld20,summed_neighbor_freq,log1p_snf,log1p_freq,consistency,reading_entropy
0,研究,けんきゅう,2,4,33,217,1296,1.00,0.90,14186,9.560081,11.192293,0.997455,0.018887
1,国語,こくご,2,3,366,110,937,1.00,1.00,586389,13.281740,8.049746,0.580080,1.078175
2,会社,かいしゃ,2,3,131,248,1140,1.00,0.85,585004,13.279376,11.704892,0.962212,0.192771
3,データ,でえた,3,3,13,14,829,1.25,1.05,1546,7.344073,9.794677,NaN,NaN
4,コース,こうす,3,3,49,689,1565,1.00,0.90,69514,11.149298,9.280706,NaN,NaN
5,独立行政法人,どくりつぎょうせいほうじん,6,12,0,0,2,3.90,6.25,0,0.000000,0.000000,0.688135,0.912249
6,発表,はっぴょう,2,4,138,45,838,1.00,1.00,417962,12.943148,11.673589,0.546663,1.070369
7,熱心,ねっしん,2,4,209,50,960,1.00,1.00,164342,12.009711,6.837333,0.678732,0.758199
8,生活,せいかつ,2,4,213,82,1343,1.00,0.95,156362,11.959936,11.209426,0.796573,0.718596
9,先生,せんせい,2,4,478,313,1382,1.00,0.60,237042,12.375997,10.233510,0.684339,0.878380


In [4]:
from psycholing_metrics_ja import predict_nonword, predict_nonword_topn

In [5]:
test_pairs = [
    ('研究', 'ケンキュウ'),
    ('国語', 'コクゴ'),
    ('今日', 'キョウ'),       # 多読み語
    ('今日', 'コンニチ'),     # 同語の別読み
    ('明日', 'アシタ'),       # 多読み語
    ('明日', 'ミョウニチ'),   # 同語の別読み
]
df = engine.compute_batch(test_pairs, verbose=False)
print(df[['word', 'reading_normalized', 'ons', 'pns', 'old20',
          'consistency', 'reading_entropy']].to_string())


  word reading_normalized  ons  pns  old20  consistency  reading_entropy
0   研究              けんきゅう   33  217    1.0     0.997455         0.018887
1   国語                こくご  366  110    1.0     0.580080         1.078175
2   今日                きょう  284  553    1.0     0.581510         1.203766
3   今日               こんにち  284   18    1.0     0.581510         1.203766
4   明日                あした  365  112    1.0     0.587479         1.277049
5   明日              みょうにち  365   11    1.0     0.587479         1.277049


In [9]:
grade6_nonwords = ['世番', '中夕', '丸当', '二右', '二歌', '公月', '内古', '円子', '前曜', '力名', '千合', '千気', '南女', '南心', '南算', '原立', '受午', '台牛', '名午', '図園', '図犬', '土店', '地王', '地町', '場天', '売京', '夏活', '夕馬', '外体', '外帯', '外本', '多形', '夜牛', '安刀', '小先', '川生', '年才', '当数', '後王', '心十', '悪土', '成週', '手海', '文出', '文血', '春金', '時兄', '曜入', '書輪', '朝字', '朝算', '木春', '来息', '来話', '東通', '歌品', '母土', '水話', '海記', '消門', '火通', '父計', '理道', '白計', '直字', '相門', '着書', '社園', '秋世', '秋国', '空字', '竹星', '竹社', '竹角', '節長', '糸文', '肉事', '茶車', '落知', '薬先', '親半', '語家', '谷作', '走筆', '足時', '足雨', '近夕', '通札', '通相', '週書', '道回', '都作', '長午', '雨円', '雪力', '音気', '風売', '風食', '食分', '麦者']

In [14]:
from psycholing_metrics_ja import predict_nonword
#predict_nonword('戦争', engine_psylex.kanji_dist).reading

C0 = [(w, predict_nonword(w, engine.kanji_dist).reading) for w in grade6_nonwords]
# C1 = [(w, predict_nonword(w, engine_wlsp.kanji_dist).reading) for w in grade6_nonwords]

# C0 = [(w, predict_nonword(w, engine_psylex.kanji_dist).reading) for w in fushimi1999_df['item'].to_list()]
# C1 = [(w, predict_nonword(w, engine_wlsp.kanji_dist).reading) for w in fushimi1999_df['item'].to_list()]
print(C0)
# print(C1)

print(engine.compute_batch(C0, verbose=True))

[('世番', 'せばん'), ('中夕', 'ちゅうゆう'), ('丸当', 'まるとう'), ('二右', 'にう'), ('二歌', 'にか'), ('公月', 'こうがつ'), ('内古', 'ないこ'), ('円子', 'えんこ'), ('前曜', 'ぜんよう'), ('力名', 'りょくめい'), ('千合', 'ちごう'), ('千気', 'ちき'), ('南女', 'なんじょ'), ('南心', 'なんしん'), ('南算', 'なんさん'), ('原立', 'げんりつ'), ('受午', 'うご'), ('台牛', 'だいぎゅう'), ('名午', 'めいご'), ('図園', 'はかえん'), ('図犬', 'はかいぬ'), ('土店', 'どてん'), ('地王', 'ちおう'), ('地町', 'ちちょう'), ('場天', 'ばてん'), ('売京', 'うきょう'), ('夏活', 'なつかつ'), ('夕馬', 'ゆうま'), ('外体', 'がいたい'), ('外帯', 'がいたい'), ('外本', 'がいほん'), ('多形', 'おおけい'), ('夜牛', 'やぎゅう'), ('安刀', 'あんとう'), ('小先', 'こせん'), ('川生', 'がわせい'), ('年才', 'ねんさい'), ('当数', 'とうすう'), ('後王', 'ごおう'), ('心十', 'しんじゅう'), ('悪土', 'わるど'), ('成週', 'せいしゅう'), ('手海', 'てかい'), ('文出', 'ぶんで'), ('文血', 'ぶんけつ'), ('春金', 'しゅんきん'), ('時兄', 'じきょうだ'), ('曜入', 'ようにゅう'), ('書輪', 'しょりん'), ('朝字', 'ちょうじ'), ('朝算', 'ちょうさん'), ('木春', 'きしゅん'), ('来息', 'らいむす'), ('来話', 'らいわ'), ('東通', 'ひがしつう'), ('歌品', 'かひん'), ('母土', 'ぼど'), ('水話', 'すいわ'), ('海記', 'かいき'), ('消門', 'しょうもん'), ('火通', 'かつう'), ('父計', 'ちちけい'), ('理道', 'りどう'), ('白計', 'はく

In [15]:
# %%time
# C1_df = engine_wlsp.compute_batch(C1, verbose=True)
# C1_df

## 概要

`psycholing_metrics_ja.py` は、日本語の単語および非単語に対して以下の心理言語学変数を一括計算する Python モジュールである。

| 変数 | 略称 | 定義 |
|------|------|------|
| 書記近傍サイズ | ONS | 見出し本体（表記）の 1 文字置換で到達可能な語の数 |
| 音韻近傍サイズ | PNS | 読み（モーラ列）の 1 モーラ置換で到達可能な語の数 |
| 書記素 Levenshtein 距離 | OLD20 | 表記が最も近い 20 語との平均編集距離 |
| 読みの一貫性 | consistency | 漢字ごとの最頻読みの確率。語全体では構成漢字の平均 |
| 読みのエントロピー | reading_entropy | 漢字ごとの読み分布のエントロピー。語全体では構成漢字の平均 |

注：ONS, OLD20 の「書記素（orthographic）」は Coltheart's N および Yarkoni et al. (2008) の原語に対応する訳語として使用している。本モジュールが分析対象とする書記素–音韻対応の一貫性（spelling-to-sound consistency）における「書記素（graphemic）」とは異なる術語レベルの概念である。

すべての計算は、国立国語研究所『分類語彙表増補改訂版データベース』（WLSP）を母集団として用いる。読みは WLSP の規則に従い、長音符号「ー」を母音に解決し、カタカナをひらがなに変換してから処理される。

---

## 動作要件

- Python 3.9 以上
- pandas
- numpy
- python-Levenshtein（推奨。インストールされていない場合は Pure Python 実装にフォールバックする。OLD20 の計算速度が約 10 倍遅くなる）

```bash
pip install pandas numpy python-Levenshtein
```

別途、WLSP データファイル `bunruidb.txt` が必要である。以下から取得する。

- https://github.com/masayu-a/WLSP

---

## 基本的な使い方

### 1. エンジンの初期化

```python
from psycholing_metrics_ja import PsychoLingEngine

engine = PsychoLingEngine(
    wlsp_path='path/to/bunruidb.txt',  # WLSP データファイル
    em_n_iter=20,                       # EM反復回数（デフォルト: 20）
    em_max_chars=4,                     # EM学習対象の最大文字数（デフォルト: 4）
)
```

初期化時に以下の処理が自動的に行われる。

1. WLSP の読み込み（101,067 レコード）
2. 書記素近傍インデックスの構築（ONS 用）
3. 音韻近傍インデックスの構築（PNS 用）
4. EM アルゴリズムによる漢字–読み対応の学習（consistency 用）

初期化には約 30 秒かかる。学習済みのエンジンは再利用できる。

### 2. 単語の心理言語学変数を計算する

```python
result = engine.compute("研究", "ケンキュウ")

print(result.ons)              # 12
print(result.pns)              # 48
print(result.old20)            # 1.0
print(result.consistency)      # 0.922
print(result.reading_entropy)  # 0.302
print(result.n_chars)          # 2
print(result.n_morae)          # 4
print(result.reading_normalized)  # けんきゅう
```

`compute()` は `WordMetrics`（NamedTuple）を返す。各フィールドは以下の通り。

| フィールド | 型 | 内容 |
|------------|------|------|
| `word` | str | 入力語 |
| `reading_normalized` | str | WLSP 方式に正規化された読み（ひらがな、長音解決済み） |
| `n_chars` | int | 文字数 |
| `n_morae` | int | モーラ数 |
| `ons` | int | 書記素近傍サイズ |
| `pns` | int | 音韻近傍サイズ |
| `old20` | float | 書記素 Levenshtein 距離 20 |
| `consistency` | float | 構成漢字の読み一貫性の平均（漢字を含まない語は NaN） |
| `reading_entropy` | float | 構成漢字の読みエントロピーの平均（漢字を含まない語は NaN） |

読みの入力はカタカナ、ひらがな、長音符号付きのいずれでもよい。内部で WLSP 方式に自動正規化される。

### 3. 複数語を一括計算する

```python
pairs = [
    ("研究", "ケンキュウ"),
    ("会社", "カイシャ"),
    ("データ", "データ"),
    ("発表", "ハッピョウ"),
]

df = engine.compute_batch(pairs, verbose=True)
print(df)
```

`compute_batch()` は pandas DataFrame を返す。列名は `WordMetrics` のフィールド名と同一。100語ごとに進捗を表示する（`verbose=False` で抑制可能）。

### 4. 結果を CSV に保存する

```python
df.to_csv("psycholing_results.csv", index=False)
```

---

## 漢字の読み分布を調べる

### 単一漢字の読み分布を表示する

```python
from psycholing_metrics_ja import show_kanji_readings

show_kanji_readings('生', engine.kanji_dist, top_n=5)
```

出力例：

```
漢字 '生' — 読み分布 (total=702.1):
  せい           382.9  ( 54.5%)
  い             74.0  ( 10.5%)
  しょう           69.7  (  9.9%)
  なま            60.0  (  8.5%)
  う             45.1  (  6.4%)
```

注：EM の学習対象に漢字かな混じり語（送り仮名付き語）が含まれるため、訓読み「い」（生きる）、「う」（生む）等も分布に反映される。

### 単一漢字の統計量を取得する

```python
from psycholing_metrics_ja import kanji_stats

st = kanji_stats('生', engine.kanji_dist)
print(st.consistency)   # 0.545 — 最頻読み「せい」の確率
print(st.entropy)       # 1.590 — 読み分布のエントロピー（nats）
print(st.n_readings)    # 5 — 確率1%以上の読みの数
print(st.top_reading)   # ('せい',) — 最頻読み（モーラtuple）
print(st.top_prob)      # 0.545 — 最頻読みの確率
```

### 全漢字の一貫性辞書を取得する

```python
from psycholing_metrics_ja import build_consistency_from_dist

cons_dict = build_consistency_from_dist(engine.kanji_dist)
print(cons_dict['会'])  # 0.887
print(cons_dict['発'])  # 0.426
```

---

## 漢字–読みアライメントを調べる

EM アルゴリズムが学習したアライメント（漢字と読みモーラの対応関係）を表示する。

```python
from psycholing_metrics_ja import show_alignment

show_alignment('発表', 'はっぴょう',
               engine.alignments, engine.confidences, engine.align_logliks)
```

出力例：

```
発→はっ  表→ぴょう  (発表/はっぴょう)  conf=1.000  loglik=-0.123
```

漢字かな混じり語のアライメントも表示できる。

```python
show_alignment('落とす', 'おとす',
               engine.alignments, engine.confidences, engine.align_logliks)
```

出力例：

```
落→お  と→と  す→す  (落とす/おとす)  conf=1.000  loglik=-0.215
```

注意: `show_alignment()` のキーは `(見出し本体, 読み)` のペアである。EM学習に使用されたデータのキーと一致する必要がある。WLSPの読みはひらがなであるため、カタカナで呼ぶと "not found" になる。

`engine.alignments` は辞書であり、キーの一覧は以下で確認できる。

```python
list(engine.alignments.keys())[:10]
```

---

## 非単語の読みを予測する

EM アルゴリズムが学習した漢字ごとの読み分布を用いて、非単語の読みを予測する。これは CDP モデルにおける非語彙経路（GPC 層）の近似に相当する。

### 最尤読みを取得する

```python
from psycholing_metrics_ja import predict_nonword

result = predict_nonword('熱校', engine.kanji_dist)
print(result.reading)           # ねつこう
print(result.per_kanji)         # (('ねつ',), ('こう',))
print(result.joint_prob)        # 0.6680
print(result.word_consistency)  # 0.829
print(result.word_entropy)      # 0.492
```

`predict_nonword()` は `NonwordReading`（NamedTuple）を返す。各フィールドは以下の通り。

| フィールド | 型 | 内容 |
|------------|------|------|
| `reading` | str | 最尤読み（モーラ連結文字列） |
| `per_kanji` | tuple of tuple | 漢字ごとの最尤読み（モーラtuple） |
| `joint_prob` | float | 漢字ごとの最尤確率の積 |
| `joint_logprob` | float | joint_prob の対数 |
| `per_kanji_consistency` | tuple of float | 漢字ごとの一貫性 |
| `per_kanji_entropy` | tuple of float | 漢字ごとのエントロピー |
| `word_consistency` | float | 語全体の一貫性（漢字平均） |
| `word_entropy` | float | 語全体のエントロピー（漢字平均） |

### 上位N候補を取得する

```python
from psycholing_metrics_ja import predict_nonword_topn

candidates = predict_nonword_topn('熱校', engine.kanji_dist, top_n=5)
for reading, per_kanji, prob in candidates:
    print(f"  {reading}  P={prob:.4f}")
```

出力例：

```
  ねつこう  P=0.6680
  ねっこう  P=0.2033
  あつこう  P=0.0987
  ねつきょう  P=0.0082
  ねつあぜ  P=0.0082
```

候補は漢字ごとの上位読みの直積から生成される。

### 整形表示する

```python
from psycholing_metrics_ja import show_nonword_prediction

show_nonword_prediction('熱校', engine.kanji_dist)
```

漢字ごとの読み・一貫性・エントロピー、結合確率、上位5候補を一括表示する。

---

## 読みの正規化（低レベル API）

エンジンを使わずに、読みの正規化やモーラ分解を単独で行うことができる。

### 長音の母音化

```python
from psycholing_metrics_ja import resolve_chouon

resolve_chouon('データ')        # 'デエタ'
resolve_chouon('コンピューター')  # 'コンピュウタア'
resolve_chouon('ティー')        # 'ティイ'
```

WLSP規則に基づき、長音符号「ー」を直前の仮名の母音段に解決する。直前が特殊拍（ン、ッ）の場合はさらに遡る。先頭のーや解決不能なーはそのまま保持される。

### カタカナ→ひらがな変換

```python
from psycholing_metrics_ja import kata_to_hira

kata_to_hira('データ')  # 'でえた'  （ーが残っている場合はそのまま）
```

### 一括正規化（長音解決 + ひらがな変換）

```python
from psycholing_metrics_ja import normalize_reading

normalize_reading('データ')        # 'でえた'
normalize_reading('コンピューター')  # 'こんぴゅうたあ'
```

### モーラ分解（ひらがな用）

```python
from psycholing_metrics_ja import tokenize_mora

tokenize_mora('けんきゅう')    # ('け', 'ん', 'きゅ', 'う')
tokenize_mora('ぷれっしゃあ')  # ('ぷ', 'れ', 'っ', 'しゃ', 'あ')
```

ひらがな正規化済みの読みを入力とする。拗音（`きゃ`, `しゅ` 等）は1モーラ、撥音（`ん`）・促音（`っ`）は各1モーラとして扱う。

### モーラ分解（カタカナ/ひらがな混在対応）

```python
from psycholing_metrics_ja import kana_to_mora

kana_to_mora('シャンプーハット')
# ['シャ', 'ン', 'プ', 'ー', 'ハ', 'ッ', 'ト']

kana_to_mora('センセイ')
# ['セ', 'ン', 'セ', 'イ']
```

EMアルゴリズムの内部で使用されるモーラ分解関数であり、長音符号「ー」は解決せずそのまま1モーラとして扱う。

---

## 設計上の注意点

### ONS/PNS のカウント方式

本モジュールは Coltheart's N の定義に従い、重複排除（unique）でカウントする。
WLSP に同一語形が複数レコード存在しても 1 回だけカウントする。JALEX データベース（Ota & Mochizuki, 2025）は重複許容（with_duplicates）でカウントしている可能性があるため、JALEX の公式 ONS/PNS 値と本モジュールの計算値は一致しない。

### consistency の対象

EM アルゴリズムは、漢字を 1 字以上含み、漢字とひらがなのみで構成される語（2 文字以上）を学習対象とする。
これにより、全漢字語（「生活」「発表」等）に加えて、送り仮名付きの動詞・形容詞（「落とす」「難しい」「読み込む」等）もEM学習に含まれ、訓読みが読み分布に正しく反映される。

カタカナ語（例: データ、コース）、ひらがなのみの語（例: だけど）には漢字が含まれないため、consistency/entropy は計算されない（NaN）。

`em_max_chars` パラメータで学習対象の最大文字数を制限できる（デフォルト: 4）。5 文字以上の語ではモーラ分割の組合せが増加するため、計算時間が延びる。
ただし、主要な訓読みは 2〜4 文字の送り仮名付き語から既に学習されるため、デフォルト値で実用上十分である。

### WLSP データの「・」を含む項目

WLSP には「奇跡・奇蹟」のような「・」で異表記を併記した見出し本体が 935 件、「なにか・なんか」のような異読みを併記した読みが 1,439 件存在する。
本モジュールでは読みの「・」に対して先頭の読み（代表形）のみを使用する。

---

## モジュール構成

| カテゴリ | 公開API | 用途 |
|----------|---------|------|
| エンジン | `PsychoLingEngine` | 初期化・変数計算の統合クラス |
| エンジン | `WordMetrics` | 計算結果の NamedTuple |
| 正規化 | `resolve_chouon()` | 長音→母音 |
| 正規化 | `kata_to_hira()` | カタカナ→ひらがな |
| 正規化 | `normalize_reading()` | 上記2つの一括適用 |
| 正規化 | `primary_yomi()` | ・の代表形抽出 |
| モーラ | `tokenize_mora()` | ひらがな→モーラtuple |
| モーラ | `kana_to_mora()` | カタカナ/ひらがな混在→モーラlist |
| EM | `em_align()` | EMアライメント学習 |
| EM | `_build_entries()` | 学習データのエントリ構築 |
| 統計 | `kanji_stats()` | 漢字別の読み統計 |
| 統計 | `build_consistency_from_dist()` | 全漢字の一貫性辞書 |
| 統計 | `word_consistency()` | 語全体の一貫性 |
| 非単語 | `predict_nonword()` | 最尤読み予測 |
| 非単語 | `predict_nonword_topn()` | 上位N候補 |
| 表示 | `show_kanji_readings()` | 読み分布表示 |
| 表示 | `show_alignment()` | アライメント表示 |
| 表示 | `show_nonword_prediction()` | 非単語予測の整形表示 |
| 定数 | `SMALL_KANA`, `LONG_VOWEL`, `SOKUON`, `HATSUON` | モーラ分解用定数 |
| 定数 | `KanjiStats` | 漢字統計の NamedTuple |
| 定数 | `NonwordReading` | 非単語予測結果の NamedTuple |

---

## 計算時間の目安

| 処理 | 時間 |
|------|------|
| 初期化（EM 学習含む） | 約 30 秒 |
| `compute()` 1語あたり | 約 30 ms（OLD20 がボトルネック） |
| `compute_batch()` 5,736 語 | 約 3 分 |
| `predict_nonword()` 1 語あたり | <1 ms |

`python-Levenshtein` がインストールされていない場合、OLD20 の計算が約 10 倍遅くなる。

---

## 参考文献

- 国立国語研究所 (2004)『分類語彙表増補改訂版データベース』(ver.1.0). https://github.com/masayu-a/WLSP
- Ota, N. & Mochizuki, M. (2025). JALEX: Japanese version of lexical decision database. *Frontiers in Language Sciences*, 3, 1506509.
- Coltheart, M., Davelaar, E., Jonasson, J. T., & Besner, D. (1977). Access to the internal lexicon. In S. Dornic (Ed.), *Attention and performance VI*.
- Glushko, R. J. (1979). The organization and activation of orthographic knowledge in reading aloud. *Journal of Experimental Psychology: Human Perception and Performance*, 5, 674–691.
- Jared, D., McRae, K., & Seidenberg, M. S. (1990). The basis of consistency effects in word naming. *Journal of Memory and Language*, 29, 687–715.
- Jared, D. (2002). Spelling-sound consistency and regularity effects in word naming. *Journal of Memory and Language*, 46, 723–750.
- Perry, C., Ziegler, J. C., & Zorzi, M. (2007). Nested incremental modeling in the development of computational theories: the CDP+ model of reading aloud. *Psychological Review*, 114, 273–315.
- Perry, C., Ziegler, J. C., & Zorzi, M. (2010). Beyond single syllables: Large-scale modeling of reading aloud with the Connectionist Dual Process (CDP++) model. *Cognitive Psychology*, 61, 106–151.
- Yarkoni, T., Balota, D., & Yap, M. (2008). Moving beyond Coltheart's N: A new measure of orthographic similarity. *Psychonomic Bulletin & Review*, 15, 971–979.
- Ziegler, J. C., Stone, G. O., & Jacobs, A. M. (1997). What is the pronunciation for -ough and the spelling for /u/? A database for computing feedforward and feedback consistency in English. *Behavior Research Methods, Instruments, & Computers*, 29, 600–618.

---

# 付録 A. Consistency の歴史的発展

英語圏（およびフランス語等のアルファベット言語）における consistency の計算方法を、歴史的発展順に整理する。

## 1. 前提: 分析単位 — body と rime

英語の1音節語は onset + body（= vowel + coda）に分解される。例：

- **MINT**: onset = /m/, body = "-int", rime = /ɪnt/
- **PINT**: onset = /p/, body = "-int", rime = /aɪnt/

Consistency は「同じ body を共有する語群（neighbors）が同じ rime を持つか」で定義される。**MINT** の body neighbors は hint, lint, tint, dint, pint 等で、このうち pint だけが異なる読み（/aɪnt/）を持つ。

## 2. Glushko (1979): 二値分類

最初期の研究では consistency を二値として扱った。同じスペリングを持つすべての語が同じ発音を共有する場合のみ consistent とし、一つでも異なる発音の語（enemy）があれば inconsistent と分類した。

つまり：
- **consistent**: enemies = 0（例: FACE — race, lace, pace すべて同じ発音）
- **inconsistent**: enemies ≥ 1（例: PINT — mint, hint, lint と衝突）

## 3. Jared, McRae & Seidenberg (1990): 連続変数化 — friends/enemies

この二値的な見方は一貫性の程度を無視しており、enemy がごく少数の語と多数の語を同列に扱っていた。現代の枠組みでは、consistency は段階的な効果を持つ連続変数として扱われる。

ここで **type consistency** の計算式が確立された：

$$C_{\text{type}} = \frac{N_{\text{friends}}}{N_{\text{friends}} + N_{\text{enemies}}}$$

- **friends**: 同じ body を持ち、同じ発音をする語の数
- **enemies**: 同じ body を持つが、異なる発音をする語の数

例（body = "-int"）:
- friends: mint, hint, lint, tint, dint, ... → 仮に8語
- enemies: pint → 1語
- C_type = 8 / (8 + 1) = 0.889

さらに Jared et al. (1990) は **token consistency**（頻度重み付き）も提案した：

$$C_{\text{token}} = \frac{\sum_{i \in \text{friends}} \text{freq}_i}{\sum_{i \in \text{friends}} \text{freq}_i + \sum_{j \in \text{enemies}} \text{freq}_j}$$

高頻度の enemy（例: PINT が高頻度なら）は consistency を大きく下げる。

注：ここでの type/token は Jared et al. (1990) の用語であり、「語の種類で数える（type）か語の出現頻度で重みづけする（token）か」の区別を指す。本マニュアル本文の「重複排除（unique）／重複許容（with_duplicates）」（WLSPの同一語形レコードを1回と数えるか複数回と数えるかの区別）とは異なる概念である。

## 4. Ziegler, Stone & Jacobs (1997): 双方向の一貫性

feedforward consistency（スペリング→音）だけでなく、feedback consistency（音→スペリング）も意味があるという提案がなされた。例えば "roar" はfeedforward consistent だが、/ɔr/ が "-ore" でも綴られうるため feedback inconsistent である。

- **Feedforward**: body "-oar" → /ɔr/ は一意か？ → Yes → feedforward consistent
- **Feedback**: /ɔr/ → "-oar" 以外に "-ore", "-oor" 等 → No → feedback inconsistent

計算式は同じ friends/enemies の比率だが、方向が逆になる。

## 5. 下位音節レベルの分解

フィードフォワードとフィードバックの consistency は、単音節語と多音節語の双方について、異なる下位音節セグメント（onset, nucleus, coda, oncleus, rime）で計算される。

1音節語 "FACE" = /feɪs/ の場合：
- onset = "f" → /f/
- nucleus = "a" → /eɪ/
- coda = "ce" → /s/
- rime = "ace" → /eɪs/
- body = "ace" → /eɪs/

各レベルで独立に friends/enemies を数え、consistency を算出できる。

## 6. EM アルゴリズムを用いた提案手法との対応

| 英語の伝統的方法 | 我々のEM手法 |
|---|---|
| 分析単位 = body (orthographic) | 分析単位 = 漢字 (graphemic) |
| neighbors = 同じ body を持つ語群 | neighbors = 同じ漢字を含む語群 |
| friends = 同じ body→rime マッピング | friends = 同じ漢字→読みマッピング |
| C = friends / (friends + enemies) | C = max P(r\|K) |
| type vs token = 頻度重みなし/あり | freq_weights=None vs word_freq |

数学的に等価性を示す。ある漢字 K について、読み $r_1$ を持つ語が $n_1$ 個、$r_2$ を持つ語が $n_2$ 個あるとき：

- friends（$r_1$ の視点）= $n_1$、enemies = $n_2$
- $\displaystyle C_{\text{type}} = \frac{n_1}{n_1 + n_2}$

これは我々の `max(count) / total` と同一。**EM が正しくアラインメントしている限り、提案手法の consistency は英語文献の type consistency の一般化になっている。**

本質的な違いは一点だけ。英語では body の定義が言語学的に先験的に決まっている（onset-rime 分割は音韻論から与えられる）のに対し、日本語の漢字では「どこからどこまでがこの漢字の読みか」が自明でないため、EM によるアラインメントが必要になる。

---

# 付録 B. 他言語におけるアラインメント問題

日本語以外の言語でも、書記素と音韻の対応関係についての問題は存在する。
このアラインメント問題の構造は言語ごとに異なるので、以下で整理する。

## 言語類型と alignment 問題の所在

### 1. アラインメント不要 — 書記素と音韻の対応が先験的に定まる言語

**英語（単音節語）**: onset-rime 分割が音韻論から一意に決まる。body "-int" → rime /ɪnt/ の対応は書記素の境界が明示的。consistency は friends/enemies を数えるだけ。

**中国語**: 1 文字＝1 音節の対応が原則的に成り立つ。音声旁（phonetic radical）が文字の発音を示し、consistency は同じ音声旁を持つ文字群が同じ発音を共有する程度として定義される。
つまり分析単位は「音声旁」であり、アラインメントは不要。ただし consistency の定義自体は我々の手法と同型（max P(pronunciation|radical) / total）。

**イタリア語・フィンランド語等の浅い書記素**: 書記素—音素対応がほぼ1対1なので、アラインメントも consistency もほとんど問題にならない。

### 2. アラインメントが必要 — 日本語と構造的に類似する問題を持つ言語

**英語（多音節語）**: ここが重要。単音節語では body-rime 分析が機能するが、多音節語では**音節境界がどこにあるか**が自明でなくなる。Jared (2002) は2音節語の consistency を body-onset-body (BOB) セグメントで分析したが、orthographic body の境界をどこに置くかは推定が必要になる。CDP++（Perry, Ziegler & Zorzi, 2010）が多音節語に拡張された際、まさに音節分割とストレス付与の学習がモデルの中核課題になった。

**ヘブライ語（非母音表記）**: ヘブライ語とアラビア語はアブジャド（子音主体の文字体系）を使用し、母音は任意のダイアクリティカルマークで表記される。母音が書かれないため、学習者は音声的曖昧性、広範な同形異義語、形態論的不予測性に直面する。

これは日本語とは異なる種類のアラインメント問題だが、本質は同じ。子音列 K-T-B から完全な音韻表象 /katav/（書いた）を復元するには、形態論的テンプレート（root-and-pattern）との照合が必要。
pointed Hebrew は補足的母音記号を用い、初学者に一貫した音韻的に十分指定されたスクリプトを提供する一方、成人読者は母音なしの unpointed Hebrew の大量の同形異義語に対処しなければならない。つまり orthography → phonology の mapping に**形態論的知識に基づく推定**が介在する。

**アラビア語**: ヘブライ語と同様のアブジャド問題に加え、接辞・屈折が豊富で、同一子音列から複数の語形が派生する。ヘブライ語は形態素が線形・非線形の両原理で統合される豊かで複雑なセム語形態論システムを特徴とする——root-and-pattern morphology である。この root-and-pattern 構造は、母音パターンが子音骨格に**非連接的に（discontinuously）挿入される**ため、我々の EM 的な確率的アラインメントと構造的に近い問題を生む。

**タイ語**: 語境界がスペースで示されない上に、子音クラスターの読み規則が文脈依存的。書記素列から音韻列への対応が多対多になる場面がある。

**チベット語**: 歴史的な音韻変化により、現代の発音と書記素の乖離が極めて大きい。7 世紀の書記素が保存されたまま、発音が大幅に変化した。

### 3. 問題構造の比較

| 言語 | アラインメント問題の種類 | 我々の手法の適用可能性 |
|------|---|---|
| 日本語（漢字熟語） | N 文字 → M モーラの分割点推定 | **直接適用可能（本研究）** |
| 英語（多音節語） | 音節境界・ストレス位置の推定 | 適用可能（書記素→音素の分割に EM を使える） |
| ヘブライ語 | 子音骨格 → 母音パターンの復元 | **構造が異なる**。分割ではなく挿入の問題。root + template の組合せ最適化が必要 |
| アラビア語 | ヘブライ語と同型 + ダイグロシア | 同上 |
| タイ語 | 語境界推定 + 子音クラスター読み | 分割問題として定式化可能 |

## 核心的な違い

日本語と英語多音節語の問題は **線形分割**（連続するモーラ/音素列をどこで切るか）だから、我々の EM 手法がそのまま適用できる。

ヘブライ語・アラビア語の問題は**非線形挿入**（子音骨格の間に母音パターンを挿む）なので、分割点の EM 推定では対処できない。
root K-T-B に対して template CaCaC → katab, CiCCeC → kitbel のような離散的テンプレート選択の推定が必要になり、HMM や CRF のような系列ラベリング手法の方が適合する。

つまり、我々の EM アルゴリズムを用いた提案手法が直接一般化できるのは **書記素と音韻の対応が連接的（concatenative）である言語** に限られる。セム語のような非連接的形態論を持つ言語には、異なるアーキテクチャが必要になる。
